In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder

In [ ]:
DATASET_PATH = r"weatherAUS.csv"
TARGET_COLUMN = "RainTomorrow"

df = pd.read_csv(DATASET_PATH)
print(f"Loaded dataset shape (all columns): {df.shape}")
df.head()

In [ ]:
# Keep rows with a valid target and optionally cap data size for faster experiments.
df = df.dropna(subset=[TARGET_COLUMN]).copy()

SUBSET_SIZE = 50000  # set to None to use all available rows
if SUBSET_SIZE is not None:
    df = df.sample(n=min(SUBSET_SIZE, len(df)), random_state=42).reset_index(drop=True)
else:
    df = df.reset_index(drop=True)

# Convert target to 0/1 for classification.
df[TARGET_COLUMN] = df[TARGET_COLUMN].replace({"Yes": 1, "No": 0}).astype("int64")

print(f"Shape after target cleaning & sizing: {df.shape}")
print(f"Target distribution:\n{df[TARGET_COLUMN].value_counts()}")

In [ ]:
feature_columns = [column for column in df.columns if column != TARGET_COLUMN]

X_raw = df[feature_columns].copy()
y = df[TARGET_COLUMN].to_numpy(dtype=np.int64)

numeric_columns = X_raw.select_dtypes(include=[np.number]).columns.tolist()
categorical_columns = [column for column in feature_columns if column not in numeric_columns]

numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numeric_columns),
        ("cat", categorical_pipeline, categorical_columns),
    ]
)

print(f"Total raw feature columns: {len(feature_columns)}")
print(f"Numeric feature columns: {len(numeric_columns)}")
print(f"Categorical feature columns: {len(categorical_columns)}")

In [ ]:
X_processed = preprocessor.fit_transform(X_raw)

print(f"Processed feature matrix shape: {X_processed.shape}")
print(f"Target vector shape: {y.shape}")

In [ ]:
# Scale all processed features to [0, 1], then map to [0, π] for quantum rotation angles.
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X_processed)
X_scaled = X_scaled * np.pi

print(f"Scaled feature matrix shape: {X_scaled.shape}")
print(f"Feature range after scaling: [{X_scaled.min():.4f}, {X_scaled.max():.4f}]")

In [ ]:
pca = PCA(n_components=3, random_state=42)
X_pca = pca.fit_transform(X_scaled)

print(f"PCA output shape (3 features): {X_pca.shape}")
print(f"Explained variance ratio: {pca.explained_variance_ratio_}")
print(f"Total explained variance: {pca.explained_variance_ratio_.sum():.4f}")

In [88]:
X_train, X_test, y_train, y_test = train_test_split(
    X_pca,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

print(f"Train class distribution: {np.bincount(y_train)}")
print(f"Test class distribution:  {np.bincount(y_test)}")

Train class distribution: [31092  8908]
Test class distribution:  [7773 2227]


In [89]:
X_train = np.asarray(X_train, dtype=np.float64)
X_test = np.asarray(X_test, dtype=np.float64)
y_train = np.asarray(y_train, dtype=np.int64)
y_test = np.asarray(y_test, dtype=np.int64)

In [90]:
print(f"X_train shape: {X_train.shape}")
print(f"X_test  shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test  shape: {y_test.shape}")

X_train shape: (40000, 3)
X_test  shape: (10000, 3)
y_train shape: (40000,)
y_test  shape: (10000,)


In [91]:
print("Final dataset summary")
print(f"- Raw rows used: {len(df)}")
print(f"- Processed feature count before PCA: {X_processed.shape[1]}")
print(f"- PCA feature count: {X_pca.shape[1]}")
print(f"- Train/Test sizes: {len(y_train)} / {len(y_test)}")

Final dataset summary
- Raw rows used: 50000
- Processed feature count before PCA: 3429
- PCA feature count: 3
- Train/Test sizes: 40000 / 10000


In [92]:
# Persist prepared dataset artifacts for VQC training/reuse.
prepared_dir = "prepared_data"
os.makedirs(prepared_dir, exist_ok=True)

np.savez_compressed(
    os.path.join(prepared_dir, "vqc_prepared_dataset.npz"),
    X_train=X_train,
    X_test=X_test,
    y_train=y_train,
    y_test=y_test,
    X_pca=X_pca,
    y=y,
)

prepared_df = pd.DataFrame(X_pca, columns=["PC1", "PC2", "PC3"])
prepared_df[TARGET_COLUMN] = y
prepared_df.to_csv(os.path.join(prepared_dir, "vqc_prepared_pca_dataset.csv"), index=False)

print("Saved prepared files:")
print(f"- {os.path.join(prepared_dir, 'vqc_prepared_dataset.npz')}")
print(f"- {os.path.join(prepared_dir, 'vqc_prepared_pca_dataset.csv')}")

Saved prepared files:
- prepared_data\vqc_prepared_dataset.npz
- prepared_data\vqc_prepared_pca_dataset.csv
